# NCCT Multi-Stage Classification V1 — LVO Detection → Laterality → Etiology

Klasifikasi **bertingkat (multi-stage)** pada citra CT Non-Contrast (NCCT) pasien stroke:

```
                    ┌────────────────────────────────────────────┐
  Citra NCCT  ──►   │  STAGE 1 : Apakah pasien mengalami LVO?    │
                    └────────────────────────────────────────────┘
                              │ Ya                        │ Tidak
                              ▼                           ▼
              ┌───────────────────────────────┐      Non-LVO
              │ STAGE 2 : Laterality          │     (selesai)
              │   Left / Right                │
              ├───────────────────────────────┤
              │ STAGE 3 : Etiologi            │
              │   Cardioembolic / Non-Cardio  │
              └───────────────────────────────┘
```

- **Stage 1** — Klasifikasi biner: **LVO** vs **Non-LVO** (semua pasien)
- **Stage 2** — Klasifikasi biner: **Left** vs **Right** (hanya pasien LVO)
- **Stage 3** — Klasifikasi biner: **Cardioembolic** vs **Non-Cardioembolic** (hanya pasien LVO)

**Sumber data:**
- `LVO_Laterality_Annotations.xlsx` → label LVO & laterality
- `Stroke_Etiology_Annotations.xlsx` → label etiologi
- Folder `LVO/pXX_sub-strokeXXXX/{normal, selected_lvo, selected_ncct}`
- Folder `NON-LVO/lacunarCT-XXX`

**Mode eksekusi:**
- `colab` — download dataset penuh dari Google Drive
- `local` — memakai sample `NVO-classification-dataset-sample` (uji coba cepat)


### 1. Install & Import Library

> `torch` & `torchvision` sudah tersedia di Google Colab. Install pustaka tambahan: `gdown` (download dataset), `timm` (model pretrained), `openpyxl` (baca xlsx), `seaborn` (visualisasi).


In [5]:
!pip install -q gdown timm openpyxl seaborn scikit-learn

In [6]:
import os, re, glob, zipfile, random, time, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from IPython.display import display

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import timm

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, roc_auc_score, classification_report)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


### 2. Konfigurasi

Ubah `MODE` menjadi `'local'` jika menjalankan di komputer sendiri memakai dataset sample, atau `'colab'` jika menjalankan di Google Colab dengan dataset penuh.


In [7]:
# ============================================================
#  KONFIGURASI
# ============================================================
MODE = 'colab'      # 'colab' → download dataset dari Google Drive
                    # 'local' → pakai dataset sample di folder lokal

GDRIVE_URL = 'https://drive.google.com/drive/folders/1Qc15oCZhtvLcg8Exv4ilnYzwZKHD2ojN?usp=drive_link'
SAMPLE_DIR = 'NVO-classification-dataset-sample'   # dataset sample utk mode lokal
DATA_ROOT  = 'dataset'                              # folder hasil download (mode colab)

LVO_SUBDIR  = 'selected_ncct'   # subfolder pasien LVO yg dipakai:
                                #   'selected_ncct' → slice NCCT dgn stroke terlihat (default)
                                #   'selected_lvo'  → slice dgn overlay mask (hanya 1 citra/pasien)
                                #   'normal'        → slice sehat
IMG_SIZE    = 224
BATCH_SIZE  = 32
EPOCHS      = 15
BACKBONE    = 'resnet18'        # backbone timm: resnet18 / resnet50 / efficientnet_b0 / ...
PRETRAINED  = True              # pakai bobot ImageNet
LR          = 1e-4
PATIENCE    = 5                 # early stopping
SEED        = 42
OUTPUT_DIR  = 'outputs'         # penyimpanan model terbaik tiap stage

DOWNLOAD_METHOD = 'drive_mount'  # 'drive_mount' → mount Google Drive & copy (disarankan utk folder besar)
                                 # 'gdown'       → download via gdown (hanya utk folder < 50 file/folder)
DRIVE_FOLDER = ''                # path manual folder di Drive (opsional); kosongkan utk auto-find

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Mode:', MODE, '| Device:', device, '| Backbone:', BACKBONE)

Mode: colab | Device: cuda | Backbone: resnet18


### 3. Download Dataset dari Google Drive

Folder publik: `https://drive.google.com/drive/folders/1Qc15oCZhtvLcg8Exv4ilnYzwZKHD2ojN`

**Kenapa bukan gdown?** `gdown` hanya bisa men-download folder dengan **≤ 50 file per folder**
(`FolderContentsMaximumLimitError`). Dataset ini berisi folder pasien dengan banyak citra (> 50),
sehingga metode yang dipakai adalah **mount Google Drive + copy**:

1. Buka link dataset di browser (harus login Google yang sama dengan Colab).
2. Klik kanan folder → **"Add shortcut to Drive"** (Tambahkan pintasan ke Drive).
   → folder kini muncul di "My Drive" dan bisa diakses Colab.
3. Jalankan cell di bawah → notebook me-mount Drive, **auto-menemukan** folder tsb,
   lalu menyalinnya ke `DATA_ROOT`.

> Ingin memakai gdown (folder kecil)? Set `DOWNLOAD_METHOD = 'gdown'`.
> Alternatif manual: isi `DRIVE_FOLDER` dengan path lengkap folder di Drive.


In [8]:
def locate_data_root(root):
    """Cari folder yang berisi file annotasi xlsx (struktur bisa berbeda setelah download)."""
    for dirpath, _, filenames in os.walk(root):
        if 'LVO_Laterality_Annotations.xlsx' in filenames:
            return dirpath
    return root

def find_in_drive(base='/content/drive/MyDrive', max_depth=3):
    """Auto-cari folder dataset (berisi xlsx annotasi) di Google Drive yang sudah di-mount."""
    for root, dirs, files in os.walk(base):
        if root[len(base):].count(os.sep) > max_depth:
            dirs[:] = []
            continue
        if 'LVO_Laterality_Annotations.xlsx' in files:
            return root
    return None

if MODE == 'colab':
    if DOWNLOAD_METHOD == 'drive_mount':
        from google.colab import drive
        drive.mount('/content/drive')   # akan meminta otorisasi (klik link & masukkan token)

        src = DRIVE_FOLDER if DRIVE_FOLDER and os.path.isdir(DRIVE_FOLDER) else find_in_drive()
        if src is None:
            raise RuntimeError('Folder dataset tidak ditemukan di Drive. Pastikan sudah menambahkan '
                               '"Add shortcut to Drive" atau isi DRIVE_FOLDER secara manual.')
        print('>>> Menyalin dataset dari Drive:', src)
        shutil.copytree(src, DATA_ROOT, dirs_exist_ok=True)
    else:
        import gdown
        print('>>> Mengunduh dataset dari Google Drive (gdown) ...')
        try:
            gdown.download_folder(url=GDRIVE_URL, output=DATA_ROOT, quiet=False)
        except Exception as e:
            print('gdown gagal:', type(e).__name__, e)
            print('Folder > 50 file tidak bisa di-download gdown. Set DOWNLOAD_METHOD = "drive_mount".')
            raise
    DATA_ROOT = locate_data_root(DATA_ROOT)
else:
    DATA_ROOT = SAMPLE_DIR
    print('>>> Mode lokal — memakai dataset sample:', DATA_ROOT)

# Jika hasil download berupa file .zip, ekstrak
for z in glob.glob(os.path.join(DATA_ROOT, '**', '*.zip'), recursive=True):
    print('>>> Ekstrak:', z)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(os.path.dirname(z))
DATA_ROOT = locate_data_root(DATA_ROOT)

print('\nStruktur dataset:')
for d in sorted(os.listdir(DATA_ROOT)):
    p = os.path.join(DATA_ROOT, d)
    print(('  [DIR]  ' if os.path.isdir(p) else '  [FILE] ') + d)


Mounted at /content/drive


RuntimeError: Folder dataset tidak ditemukan di Drive. Pastikan sudah menambahkan "Add shortcut to Drive" atau isi DRIVE_FOLDER secara manual.

### 4. Membaca Annotations & Membangun Metadata

Metadata digabung dari dua file annotasi + struktur folder:

| Kolom | Sumber | Stage |
|-------|--------|-------|
| `is_lvo` | `Has LVO` (annotasi) / keanggotaan folder `NON-LVO` | Stage 1 |
| `laterality` | `Laterality (Left/Right)` | Stage 2 |
| `etiology` | `Stroke Etiology` | Stage 3 |

> Pasien di folder `NON-LVO/` (mis. `lacunarCT-001`) tidak tercatat di annotasi → otomatis **Non-LVO**.
> Pasien di folder `LVO/` dengan `Has LVO = No` juga diberi label **Non-LVO**.


In [ ]:
# Baca kedua file annotasi
lat_df  = pd.read_excel(os.path.join(DATA_ROOT, 'LVO_Laterality_Annotations.xlsx'))
etio_df = pd.read_excel(os.path.join(DATA_ROOT, 'Stroke_Etiology_Annotations.xlsx'))

print('=== LVO_Laterality_Annotations ===')
display(lat_df.head())
print('Distribusi "Has LVO"  :', lat_df['Has LVO'].value_counts().to_dict())
print('Distribusi Laterality :', lat_df['Laterality (Left/Right)'].value_counts().to_dict())

print('\n=== Stroke_Etiology_Annotations ===')
display(etio_df.head())
print('Distribusi "Stroke Etiology" :', etio_df['Stroke Etiology'].value_counts().to_dict())

In [ ]:
def build_metadata(data_root, lvo_subdir='selected_ncct'):
    """Bangun tabel metadata: 1 baris = 1 citra, lengkap label untuk 3 stage."""
    lat_df  = pd.read_excel(os.path.join(data_root, 'LVO_Laterality_Annotations.xlsx'))
    etio_df = pd.read_excel(os.path.join(data_root, 'Stroke_Etiology_Annotations.xlsx'))
    annot = lat_df.merge(etio_df, on='Subject', how='left')

    def get_annot(subject):
        row = annot[annot['Subject'] == subject]
        if len(row) == 0:
            return None
        r = row.iloc[0]
        return {
            'is_lvo':     str(r['Has LVO']).strip().lower() == 'yes',
            'laterality': r.get('Laterality (Left/Right)'),
            'etiology':   r.get('Stroke Etiology'),
        }

    records, IMG_EXT = [], ('.jpg', '.jpeg', '.png', '.bmp')

    # ---- Pasien LVO : LVO/pXX_sub-strokeXXXX/{lvo_subdir}/*.jpg ----
    lvo_root = os.path.join(data_root, 'LVO')
    if os.path.isdir(lvo_root):
        for pdir in sorted(os.listdir(lvo_root)):
            ppath = os.path.join(lvo_root, pdir)
            if not os.path.isdir(ppath):
                continue
            m = re.search(r'(sub-stroke\d+)', pdir)
            subject = m.group(1) if m else pdir
            a = get_annot(subject)
            if a is None:      # di folder LVO tapi tak ada di annotasi → anggap LVO
                a = {'is_lvo': True, 'laterality': None, 'etiology': None}
            img_dir = os.path.join(ppath, lvo_subdir)
            if not os.path.isdir(img_dir):
                print(f'  [skip] folder tidak ditemukan: {img_dir}')
                continue
            for f in sorted(os.listdir(img_dir)):
                if f.lower().endswith(IMG_EXT):
                    records.append(dict(image_path=os.path.join(img_dir, f),
                                        subject=subject, **a))

    # ---- Pasien Non-LVO : NON-LVO/lacunarCT-XXX/*.jpg ----
    nonlvo_root = os.path.join(data_root, 'NON-LVO')
    if os.path.isdir(nonlvo_root):
        for pdir in sorted(os.listdir(nonlvo_root)):
            ppath = os.path.join(nonlvo_root, pdir)
            if not os.path.isdir(ppath):
                continue
            for f in sorted(os.listdir(ppath)):
                if f.lower().endswith(IMG_EXT):
                    records.append(dict(image_path=os.path.join(ppath, f),
                                        subject=pdir, is_lvo=False,
                                        laterality=None, etiology=None))

    df = pd.DataFrame(records)
    df['label_lvo']        = df['is_lvo'].astype(int)                  # 1=LVO, 0=Non-LVO
    df['label_laterality'] = df['laterality'].map({'Left': 0, 'Right': 1})
    df['label_etiology']   = df['etiology'].map({'Non-Cardioembolic': 0, 'Cardioembolic': 1})
    return df

meta = build_metadata(DATA_ROOT, lvo_subdir=LVO_SUBDIR)
print(f'Total citra : {len(meta)}')
print(f'Total pasien: {meta["subject"].nunique()}')
display(meta.head())

### 5. Analisis Dataset

Cek distribusi kelas pada tiap stage — penting untuk menentukan apakah perlu *class weighting* saat training.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# --- Stage 1: LVO vs Non-LVO
s1 = meta['label_lvo'].map({1: 'LVO', 0: 'Non-LVO'}).value_counts().reindex(['Non-LVO', 'LVO'])
axes[0].bar(s1.index, s1.values, color=['#3498db', '#e74c3c'])
axes[0].set_title('Stage 1: LVO vs Non-LVO')

# --- Stage 2: Laterality (hanya pasien LVO)
s2df = meta[meta['label_lvo'] == 1].dropna(subset=['label_laterality'])
s2 = s2df['label_laterality'].map({0: 'Left', 1: 'Right'}).value_counts().reindex(['Left', 'Right'])
axes[1].bar(s2.index, s2.values, color=['#2ecc71', '#f39c12'])
axes[1].set_title('Stage 2: Laterality (hanya LVO)')

# --- Stage 3: Etiologi (hanya pasien LVO)
s3df = meta[meta['label_lvo'] == 1].dropna(subset=['label_etiology'])
s3 = s3df['label_etiology'].map({0: 'Non-Cardio', 1: 'Cardio'}).value_counts().reindex(['Non-Cardio', 'Cardio'])
axes[2].bar(s3.index, s3.values, color=['#9b59b6', '#1abc9c'])
axes[2].set_title('Stage 3: Etiologi (hanya LVO)')

for ax in axes:
    ax.set_ylabel('Jumlah citra')
    for p in ax.patches:
        ax.annotate(str(int(p.get_height())), (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha='center', va='bottom')
plt.tight_layout(); plt.show()

print('Jumlah PASIEN per kelas:')
print(f"  Stage 1 — LVO: {meta[meta['label_lvo']==1]['subject'].nunique()} | Non-LVO: {meta[meta['label_lvo']==0]['subject'].nunique()}")
print(f"  Stage 2 — Left: {s2df[s2df['label_laterality']==0]['subject'].nunique()} | Right: {s2df[s2df['label_laterality']==1]['subject'].nunique()}")
print(f"  Stage 3 — Cardio: {s3df[s3df['label_etiology']==1]['subject'].nunique()} | Non-Cardio: {s3df[s3df['label_etiology']==0]['subject'].nunique()}")

In [ ]:
# Contoh citra tiap kategori
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for i, (title, sub) in enumerate([
    ('LVO — selected_ncct', meta[meta['label_lvo'] == 1]),
    ('Non-LVO — lacunar',   meta[meta['label_lvo'] == 0]),
]):
    p = sub.sample(1, random_state=SEED + i)['image_path'].iloc[0]
    img = Image.open(p).convert('RGB')
    axes[i].imshow(img)
    axes[i].set_title(f'{title}\n{os.path.basename(p)}', fontsize=9)
    axes[i].axis('off')
plt.tight_layout(); plt.show()

### 6. Dataset & Dataloader — Split per Pasien

> **Penting:** split dilakukan **per pasien**, bukan per citra. Citra dari pasien yang sama tidak boleh tersebar
> ke train & test sekaligus (kebocoran data / *data leakage*), karena slice-slice satu pasien sangat mirip.

Tiap stage punya subset data berbeda:
- **Stage 1** : semua pasien (LVO + Non-LVO)
- **Stage 2** : hanya pasien LVO yang punya label laterality
- **Stage 3** : hanya pasien LVO yang punya label etiologi


In [ ]:
class StrokeImageDataset(Dataset):
    """Dataset citra stroke: 1 sample = 1 citra + label untuk satu stage."""
    def __init__(self, df, label_col, transform=None):
        self.df = df.reset_index(drop=True)
        self.label_col = label_col
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB')
        if self.transform is not None:
            img = self.transform(img)
        return img, int(row[self.label_col]), row['image_path']

In [ ]:
# Augmentasi ringan (citra medis — jangan terlalu agresif)
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def patient_level_split(df, val_frac=0.15, test_frac=0.15, seed=SEED, min_split=1):
    """Split per PASIEN (bukan per citra) agar tidak ada kebocoran antar set."""
    subjects = list(df['subject'].unique())
    rng = np.random.RandomState(seed)
    rng.shuffle(subjects)
    n = len(subjects)
    n_val  = max(min_split, int(round(n * val_frac)))
    n_test = max(min_split, int(round(n * test_frac)))
    while n_val + n_test >= n and n_val > 0 and n_test > 0:
        if n_test >= n_val: n_test -= 1
        else:               n_val  -= 1
    val_subj   = set(subjects[:n_val])
    test_subj  = set(subjects[n_val:n_val + n_test])
    train_subj = set(subjects[n_val + n_test:])
    return (df[df['subject'].isin(train_subj)],
            df[df['subject'].isin(val_subj)],
            df[df['subject'].isin(test_subj)])

def make_loaders(df, label_col, batch_size=BATCH_SIZE):
    train_df, val_df, test_df = patient_level_split(df)
    tr_ds = StrokeImageDataset(train_df, label_col, train_tf)
    va_ds = StrokeImageDataset(val_df, label_col, eval_tf)
    te_ds = StrokeImageDataset(test_df, label_col, eval_tf)
    tr_ld = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    va_ld = DataLoader(va_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    te_ld = DataLoader(te_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    return tr_ld, va_ld, te_ld

### 7. Model — CNN Pretrained (timm)

Backbone default `resnet18` (ringan & cepat). Bisa diganti `resnet50`, `efficientnet_b0`, `densenet121`, dll.
Bobot awal memakai ImageNet (`PRETRAINED=True`) lalu di-*fine-tune* pada citra NCCT.


In [ ]:
def build_model(num_classes=2, backbone=BACKBONE, pretrained=PRETRAINED):
    """Model CNN pretrained dari timm (fallback utk kompatibilitas versi timm)."""
    try:
        model = timm.create_model(backbone, pretrained=pretrained, num_classes=num_classes)
    except TypeError:
        weights = 'IMAGENET1K_V1' if pretrained else None
        model = timm.create_model(backbone, weights=weights, num_classes=num_classes)
    return model

# Sanity check arsitektur
m = build_model(2)
x = torch.randn(2, 3, IMG_SIZE, IMG_SIZE)
print('Input :', tuple(x.shape))
print('Output:', tuple(m(x).shape), '(logits 2 kelas)')

### 8. Fungsi Training & Evaluasi

- `train_model` : loop training standar (CrossEntropy + AdamW + CosineAnnealing + early stopping).
  Model terbaik (F1 val tertinggi) disimpan ke `outputs/{label}_best.pt`.
- `evaluate_model` : metrik accuracy / precision / recall / F1 / ROC-AUC pada set mana pun.


In [ ]:
def compute_metrics(y_true, y_pred, y_prob):
    return {
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall':    recall_score(y_true, y_pred, zero_division=0),
        'f1':        f1_score(y_true, y_pred, zero_division=0),
        'auc':       roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else float('nan'),
    }

@torch.no_grad()
def evaluate_model(model, loader, criterion=None, device=device):
    model.eval()
    y_true, y_pred, y_prob, loss_sum, n = [], [], [], 0.0, 0
    for xb, yb, _ in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        probs  = torch.softmax(logits, dim=1)
        y_true.extend(yb.tolist())
        y_pred.extend(probs.argmax(dim=1).tolist())
        y_prob.extend(probs[:, 1].tolist())
        if criterion is not None:
            loss_sum += criterion(logits, yb).item() * len(yb)
            n += len(yb)
    metrics = compute_metrics(y_true, y_pred, y_prob)
    if criterion is not None:
        metrics['loss'] = loss_sum / max(n, 1)
    return metrics, (y_true, y_pred, y_prob)

def train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR, patience=PATIENCE,
                label='', class_weights=None, device=device):
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.AdamW(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}

    best_f1, best_epoch, bad_epochs = -1.0, 0, 0
    save_path = os.path.join(OUTPUT_DIR, f'{label}_best.pt')

    for epoch in range(1, epochs + 1):
        model.train()
        tot_loss, tot = 0.0, 0
        for xb, yb, _ in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            tot_loss += loss.item() * len(yb)
            tot += len(yb)
        scheduler.step()

        try:
            val_metrics, _ = evaluate_model(model, val_loader, criterion=criterion)
        except ValueError:   # val set kosong (dataset terlalu kecil)
            val_metrics = {'accuracy': 0.0, 'f1': 0.0, 'loss': float('nan')}

        tl = tot_loss / max(tot, 1)
        history['train_loss'].append(tl)
        history['val_loss'].append(val_metrics.get('loss', float('nan')))
        history['val_acc'].append(val_metrics['accuracy'])
        history['val_f1'].append(val_metrics['f1'])

        print(f'[{label}] epoch {epoch:02d}/{epochs} | train_loss={tl:.4f} '
              f'| val_loss={val_metrics.get("loss", float("nan")):.4f} '
              f'| val_acc={val_metrics["accuracy"]:.4f} | val_f1={val_metrics["f1"]:.4f}')

        if val_metrics['f1'] > best_f1:
            best_f1, best_epoch, bad_epochs = val_metrics['f1'], epoch, 0
            torch.save(model.state_dict(), save_path)
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f'  >> Early stop di epoch {epoch} (best F1={best_f1:.4f} di epoch {best_epoch})')
                break

    if os.path.exists(save_path):
        model.load_state_dict(torch.load(save_path, map_location=device))
    print(f'[{label}] Selesai. Best val F1 = {best_f1:.4f} (epoch {best_epoch})')
    return model, history

### 9. Stage 1 — LVO vs Non-LVO

Model pertama menentukan apakah citra berasal dari pasien **LVO** atau **Non-LVO**.


In [ ]:
# ===== Stage 1: LVO vs Non-LVO =====
s1_df = meta[['image_path', 'subject', 'label_lvo']].rename(columns={'label_lvo': 'label'})
print('Distribusi label Stage 1:', s1_df['label'].value_counts().to_dict())

tr_ld, va_ld, te_ld = make_loaders(s1_df, 'label')
print(f'Citra  → Train: {len(tr_ld.dataset)} | Val: {len(va_ld.dataset)} | Test: {len(te_ld.dataset)}')
print(f'Pasien → Train: {tr_ld.dataset.df["subject"].nunique()} | '
      f'Val: {va_ld.dataset.df["subject"].nunique()} | Test: {te_ld.dataset.df["subject"].nunique()}')

# Class weight utk menangani imbalance antar kelas
counts = np.bincount(s1_df['label'].values)
cw1 = torch.tensor(len(s1_df) / (2 * counts), dtype=torch.float32).to(device)
print('Class weights:', cw1.tolist())

In [ ]:
model_s1 = build_model(2).to(device)
model_s1, hist_s1 = train_model(model_s1, tr_ld, va_ld, label='stage1_lvo', class_weights=cw1)

In [ ]:
test_metrics_s1, (y1t, y1p, y1prob) = evaluate_model(model_s1, te_ld)
print('=== Stage 1: LVO vs Non-LVO (Test) ===')
for k, v in test_metrics_s1.items():
    print(f'  {k:10s}: {v:.4f}')
print()
print(classification_report(y1t, y1p, labels=[0, 1], target_names=['Non-LVO', 'LVO'], zero_division=0))

plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y1t, y1p), annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-LVO', 'LVO'], yticklabels=['Non-LVO', 'LVO'])
plt.title('Stage 1 — Confusion Matrix (Test)')
plt.show()

### 10. Stage 2 — Laterality (Left vs Right)

Hanya dijalankan untuk pasien **LVO**. Menentukan lokasi stroke: sisi **kiri (Left)** atau **kanan (Right)**.


In [ ]:
# ===== Stage 2: Laterality (Left vs Right) — HANYA pasien LVO =====
s2_df = (meta[meta['label_lvo'] == 1]
             .dropna(subset=['label_laterality'])
             [['image_path', 'subject', 'label_laterality']]
             .rename(columns={'label_laterality': 'label'}))
s2_df['label'] = s2_df['label'].astype(int)
print('Distribusi label Stage 2:', s2_df['label'].value_counts().to_dict())

tr_ld2, va_ld2, te_ld2 = make_loaders(s2_df, 'label')
print(f'Citra  → Train: {len(tr_ld2.dataset)} | Val: {len(va_ld2.dataset)} | Test: {len(te_ld2.dataset)}')
print(f'Pasien → Train: {tr_ld2.dataset.df["subject"].nunique()} | '
      f'Val: {va_ld2.dataset.df["subject"].nunique()} | Test: {te_ld2.dataset.df["subject"].nunique()}')

counts2 = np.bincount(s2_df['label'].values)
cw2 = torch.tensor(len(s2_df) / (2 * counts2), dtype=torch.float32).to(device)
print('Class weights:', cw2.tolist())

In [ ]:
model_s2 = build_model(2).to(device)
model_s2, hist_s2 = train_model(model_s2, tr_ld2, va_ld2, label='stage2_laterality', class_weights=cw2)

In [ ]:
test_metrics_s2, (y2t, y2p, y2prob) = evaluate_model(model_s2, te_ld2)
print('=== Stage 2: Laterality Left/Right (Test) ===')
for k, v in test_metrics_s2.items():
    print(f'  {k:10s}: {v:.4f}')
print()
print(classification_report(y2t, y2p, labels=[0, 1], target_names=['Left', 'Right'], zero_division=0))

plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y2t, y2p), annot=True, fmt='d', cmap='Greens',
            xticklabels=['Left', 'Right'], yticklabels=['Left', 'Right'])
plt.title('Stage 2 — Confusion Matrix (Test)')
plt.show()

### 11. Stage 3 — Etiologi (Cardio vs Non-Cardio)

Hanya dijalankan untuk pasien **LVO**. Menentukan subtype stroke: **Cardioembolic** atau **Non-Cardioembolic**.


In [ ]:
# ===== Stage 3: Etiologi (Cardio vs Non-Cardio) — HANYA pasien LVO =====
s3_df = (meta[meta['label_lvo'] == 1]
             .dropna(subset=['label_etiology'])
             [['image_path', 'subject', 'label_etiology']]
             .rename(columns={'label_etiology': 'label'}))
s3_df['label'] = s3_df['label'].astype(int)
print('Distribusi label Stage 3:', s3_df['label'].value_counts().to_dict())

tr_ld3, va_ld3, te_ld3 = make_loaders(s3_df, 'label')
print(f'Citra  → Train: {len(tr_ld3.dataset)} | Val: {len(va_ld3.dataset)} | Test: {len(te_ld3.dataset)}')
print(f'Pasien → Train: {tr_ld3.dataset.df["subject"].nunique()} | '
      f'Val: {va_ld3.dataset.df["subject"].nunique()} | Test: {te_ld3.dataset.df["subject"].nunique()}')

counts3 = np.bincount(s3_df['label'].values)
cw3 = torch.tensor(len(s3_df) / (2 * counts3), dtype=torch.float32).to(device)
print('Class weights:', cw3.tolist())

In [ ]:
model_s3 = build_model(2).to(device)
model_s3, hist_s3 = train_model(model_s3, tr_ld3, va_ld3, label='stage3_etiology', class_weights=cw3)

In [ ]:
test_metrics_s3, (y3t, y3p, y3prob) = evaluate_model(model_s3, te_ld3)
print('=== Stage 3: Etiologi Cardio/Non-Cardio (Test) ===')
for k, v in test_metrics_s3.items():
    print(f'  {k:10s}: {v:.4f}')
print()
print(classification_report(y3t, y3p, labels=[0, 1], target_names=['Non-Cardio', 'Cardio'], zero_division=0))

plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y3t, y3p), annot=True, fmt='d', cmap='Purples',
            xticklabels=['Non-Cardio', 'Cardio'], yticklabels=['Non-Cardio', 'Cardio'])
plt.title('Stage 3 — Confusion Matrix (Test)')
plt.show()

### 12. Training Curves

Kurva loss & metrik validasi untuk ketiga stage.


In [ ]:
def plot_history(hist, title):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(hist['train_loss'], label='train loss', color='#e74c3c')
    ax[0].plot(hist['val_loss'],   label='val loss',   color='#3498db')
    ax[0].set_title(f'{title} — Loss'); ax[0].legend(); ax[0].set_xlabel('epoch')
    ax[1].plot(hist['val_f1'],  label='val F1',  color='#2ecc71')
    ax[1].plot(hist['val_acc'], label='val Acc', color='#f39c12')
    ax[1].set_title(f'{title} — Metrics'); ax[1].legend(); ax[1].set_xlabel('epoch')
    plt.tight_layout(); plt.show()

plot_history(hist_s1, 'Stage 1 (LVO vs Non-LVO)')
plot_history(hist_s2, 'Stage 2 (Laterality)')
plot_history(hist_s3, 'Stage 3 (Etiologi)')

### 13. Pipeline Multi-Stage — Inferensi End-to-End

`predict_pipeline` menjalankan citra melalui seluruh stage secara berantai:

1. **Stage 1** → LVO / Non-LVO
2. Jika **LVO** → lanjut **Stage 2** (Laterality) & **Stage 3** (Etiologi)
3. Jika **Non-LVO** → selesai (laterality & etiologi = `None`)


In [ ]:
def predict_pipeline(image_path, model_s1, model_s2, model_s3, device=device, tf=eval_tf):
    """Inferensi end-to-end: Stage1 → jika LVO lanjut Stage2 & Stage3."""
    img = Image.open(image_path).convert('RGB')
    x = tf(img).unsqueeze(0).to(device)
    res = {'image': image_path}
    with torch.no_grad():
        p1 = torch.softmax(model_s1(x), dim=1)[0]
    res['pred_stage1'] = 'LVO' if p1.argmax() == 1 else 'Non-LVO'
    res['p_lvo'] = p1[1].item()
    if res['pred_stage1'] == 'LVO':
        with torch.no_grad():
            p2 = torch.softmax(model_s2(x), dim=1)[0]
            p3 = torch.softmax(model_s3(x), dim=1)[0]
        res['pred_laterality'] = 'Left' if p2.argmax() == 0 else 'Right'
        res['pred_etiology']   = 'Cardioembolic' if p3.argmax() == 1 else 'Non-Cardioembolic'
    else:
        res['pred_laterality'], res['pred_etiology'] = None, None
    return res

# Contoh inferensi 1 citra dari set test Stage 1
sample_p = te_ld.dataset.df['image_path'].iloc[0]
print(predict_pipeline(sample_p, model_s1, model_s2, model_s3))

In [ ]:
# Evaluasi pipeline end-to-end pada seluruh set test (per citra)
gt = meta.set_index('image_path')
rows = []
for p in te_ld.dataset.df['image_path'].tolist():
    r = predict_pipeline(p, model_s1, model_s2, model_s3)
    g = gt.loc[p]
    is_lvo_gt = g['label_lvo'] == 1
    rows.append({
        'image_path': p,
        'gt_stage1':   'LVO' if is_lvo_gt else 'Non-LVO',
        'pred_stage1': r['pred_stage1'],
        'gt_laterality': ('Left' if g['label_laterality'] == 0 else 'Right') if is_lvo_gt else None,
        'pred_laterality': r['pred_laterality'],
        'gt_etiology': ('Cardioembolic' if g['label_etiology'] == 1 else 'Non-Cardioembolic') if is_lvo_gt else None,
        'pred_etiology': r['pred_etiology'],
    })
pipe_eval = pd.DataFrame(rows)
print(f'Jumlah citra test: {len(pipe_eval)}')

acc1 = (pipe_eval['gt_stage1'] == pipe_eval['pred_stage1']).mean()
print(f'\nStage 1 accuracy (test)          : {acc1:.4f}')

lvo_sub = pipe_eval[pipe_eval['gt_stage1'] == 'LVO']
if len(lvo_sub) > 0:
    acc2 = (lvo_sub['gt_laterality'] == lvo_sub['pred_laterality']).mean()
    acc3 = (lvo_sub['gt_etiology']    == lvo_sub['pred_etiology']).mean()
    print(f'Stage 2 accuracy (pd pasien LVO) : {acc2:.4f}  (n={len(lvo_sub)})')
    print(f'Stage 3 accuracy (pd pasien LVO) : {acc3:.4f}  (n={len(lvo_sub)})')
display(pipe_eval.head(10))

### 14. Visualisasi Prediksi Pipeline

Contoh hasil prediksi end-to-end pada set test (hijau = benar, merah = salah).


In [ ]:
def plot_predictions(pipe_eval, n=6):
    sample = pipe_eval.sample(min(n, len(pipe_eval)), random_state=SEED)
    cols = min(3, len(sample))
    rows = int(np.ceil(len(sample) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4.5 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, (_, r) in zip(axes, sample.iterrows()):
        img = Image.open(r['image_path']).convert('RGB')
        ax.imshow(img); ax.axis('off')
        t = f"GT   : {r['gt_stage1']}"
        if r['gt_laterality'] is not None:
            t += f" | {r['gt_laterality']} | {r['gt_etiology']}"
        t += f"\nPRED : {r['pred_stage1']}"
        if r['pred_laterality'] is not None:
            t += f" | {r['pred_laterality']} | {r['pred_etiology']}"
        ok = r['gt_stage1'] == r['pred_stage1']
        ax.set_title(t, fontsize=8, color='green' if ok else 'red')
    for ax in axes[len(sample):]:
        ax.axis('off')
    plt.tight_layout(); plt.show()

plot_predictions(pipe_eval)

### 15. Ringkasan & Kesimpulan

**Pipeline multi-stage selesai diimplementasikan:**

| Stage | Tugas | Kelas | Data training | Model tersimpan |
|-------|-------|-------|---------------|-----------------|
| 1 | Deteksi LVO | LVO / Non-LVO | Semua pasien | `outputs/stage1_lvo_best.pt` |
| 2 | Laterality | Left / Right | Pasien LVO saja | `outputs/stage2_laterality_best.pt` |
| 3 | Etiologi | Cardioembolic / Non-Cardioembolic | Pasien LVO saja | `outputs/stage3_etiology_best.pt` |

**Catatan penting:**
- Split data dilakukan **per pasien** untuk menghindari kebocoran data antar set.
- Dengan sample lokal (6 pasien) hasil training hanyalah demonstrasi; jalankan dengan **dataset penuh di Colab** (mode `'colab'`) untuk hasil yang valid & klinis.
- Backbone dapat diganti (`resnet50`, `efficientnet_b0`, `densenet121`, ...) lewat variabel `BACKBONE`.
- Subfolder citra pasien LVO dapat diubah lewat `LVO_SUBDIR` (`selected_ncct` / `selected_lvo` / `normal`).
- Model terbaik tiap stage tersimpan di folder `outputs/` dan siap dipakai ulang untuk inferensi.
